# Severity-Aware Emergency Dispatch Using QAOA
**IBM Qiskit Fall Fest 2026 – RV University**

> A step-by-step walkthrough of the full pipeline: problem setup → QUBO → Ising Hamiltonian → QAOA → benchmark results.

**All results are from a Qiskit Statevector Simulator — no real quantum hardware.**

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 130
import matplotlib.pyplot as plt

print('Environment ready.')

## 1 · Problem: Why Nearest-Vehicle Dispatch Fails

When several incidents of different severity compete for a limited fleet,
a **nearest-vehicle** heuristic can leave a **critical patient waiting** while
a nearby ambulance handles a minor call.

### Model
| Symbol | Meaning |
|--------|---------|
| `v = 0..n-1` | Vehicles (ambulances) |
| `i = 0..m-1` | Incidents, each with severity `s_i ∈ {1..5}` |
| `w_i` | Severity weight: `{1:1, 2:2, 3:4, 4:8, 5:16}` |
| `t[v][i]` | Manhattan travel time |
| `c[v][i] = w_i · t[v][i]` | Weighted cost |
| `x[v,i] ∈ {0,1}` | Assignment variable |

In [ ]:
from data.instance import generate_greedy_failure_instance, plot_dispatch_map
from solvers.exact import solve_hungarian
from solvers.greedy import solve_nearest_vehicle, solve_severity_first_greedy

inst = generate_greedy_failure_instance()
h_res  = solve_hungarian(inst.cost_matrix)
nv_res = solve_nearest_vehicle(inst.travel_time_matrix, inst.weights)

print('Cost matrix (rows=vehicles, cols=incidents):')
print(np.round(inst.cost_matrix, 1))
print(f'\nHungarian optimal cost : {h_res["total_cost"]:.1f}')
print(f'Nearest-Vehicle cost   : {nv_res["total_cost"]:.1f}  (ratio {nv_res["total_cost"]/h_res["total_cost"]:.2f}x)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sev_colors = {1: '#2ecc71', 2: '#f1c40f', 3: '#e67e22', 4: '#e74c3c', 5: '#8e44ad'}

def draw_scenario(ax, res, title, tcolor):
    ax.set_facecolor('#f8f9fa')
    ax.grid(True, linestyle='--', alpha=0.4)
    vp, ip = inst.vehicle_positions, inst.incident_positions
    ax.scatter(vp[:,0], vp[:,1], s=220, c='#2980b9', marker='s',
               edgecolors='black', linewidths=1.5, label='Ambulance', zorder=5)
    for v,(x,y) in enumerate(vp):
        ax.annotate(f'V{v}', (x,y), xytext=(0,-16), textcoords='offset points',
                    ha='center', fontsize=9, fontweight='bold')
    for i,(x,y) in enumerate(ip):
        sev = inst.severities[i]
        ax.scatter(x, y, s=180+sev*25, c=sev_colors[sev], marker='o',
                   edgecolors='black', linewidths=1.5, zorder=6)
        ax.annotate(f'I{i}\n(s={sev})', (x,y), xytext=(0,12),
                    textcoords='offset points', ha='center', fontsize=8, fontweight='bold')
    for v_idx, i_idx in res['assignments']:
        vxy, ixy = vp[v_idx], ip[i_idx]
        w = inst.weights[i_idx]
        ax.annotate('', xy=(ixy[0],ixy[1]), xytext=(vxy[0],vxy[1]),
                    arrowprops=dict(arrowstyle='->', lw=2.2, shrinkA=8, shrinkB=8,
                                    color='#c0392b' if w>=8 else '#27ae60'), zorder=4)
        mx,my = (vxy[0]+ixy[0])/2, (vxy[1]+ixy[1])/2
        ax.text(mx, my, f'{inst.travel_time_matrix[v_idx,i_idx]:.0f}m', fontsize=7.5,
                ha='center', bbox=dict(boxstyle='round', fc='white', ec='gray', alpha=0.8))
    ax.set_xlim(-0.5, inst.grid_size+0.5); ax.set_ylim(-0.5, inst.grid_size+0.5)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10, color=tcolor, fontweight='bold')

draw_scenario(axes[0], nv_res,
    f'Nearest-Vehicle Greedy  (Cost={nv_res["total_cost"]:.0f})', 'crimson')
draw_scenario(axes[1], h_res,
    f'Hungarian Optimal  (Cost={h_res["total_cost"]:.0f})', 'darkgreen')

fig.suptitle('Greedy Failure: Why Severity-Aware Assignment Matters',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 2 · QUBO Formulation

After padding to a square N×N problem:

$$
H(\mathbf{x}) = \sum_{v,i} c_{vi} x_{vi}
  + P \sum_v\!\left(\sum_i x_{vi} - 1\right)^2
  + P \sum_i\!\left(\sum_v x_{vi} - 1\right)^2
$$

Expanding and collecting: $H(\mathbf{x}) = \mathbf{x}^\top Q \mathbf{x} + \text{offset}$

In [ ]:
from data.instance import generate_instance
from model.qubo import build_qubo, evaluate_qubo_energy
from solvers.exact import solve_brute_force_qubo

inst3 = generate_instance(3, 3, seed=42)
Q, offset, P = build_qubo(inst3.cost_matrix)
h_res3 = solve_hungarian(inst3.cost_matrix)
bf_res = solve_brute_force_qubo(Q, offset, 3)

print(f'N = 3  →  {3*3} qubits')
print(f'Q matrix shape : {Q.shape}')
print(f'Penalty P      : {P:.2f}  (= 1.5 × max(C) = {1.5*np.max(inst3.cost_matrix):.2f})')
print(f'QUBO offset    : {offset:.2f}')
print(f'Hungarian cost : {h_res3["total_cost"]:.4f}')
print(f'Brute-force QUBO min : {bf_res["min_energy"]:.4f}')
print(f'Match          : {abs(h_res3["total_cost"] - bf_res["min_energy"]) < 1e-6}')

### Validate on 10 random instances

In [ ]:
passed = 0
for seed in range(10):
    inst_v = generate_instance(3, 3, seed=seed)
    Q_v, off_v, _ = build_qubo(inst_v.cost_matrix)
    h_v = solve_hungarian(inst_v.cost_matrix)
    bf_v = solve_brute_force_qubo(Q_v, off_v, 3)
    ok = abs(h_v['total_cost'] - bf_v['min_energy']) < 1e-6
    passed += int(ok)
    print(f'  seed={seed:2d}  Hungarian={h_v["total_cost"]:8.3f}  BruteForce={bf_v["min_energy"]:8.3f}  {"✓" if ok else "✗"}')
print(f'\nPassed: {passed}/10')

## 3 · Ising Hamiltonian

Substitute $x_k = (I - Z_k)/2$ to get:

$$H_{\text{Ising}} = \sum_k h_k Z_k + \sum_{k<l} J_{kl} Z_k Z_l + c \cdot I$$

Built as a Qiskit `SparsePauliOp`.

In [ ]:
from model.ising import qubo_to_ising, decode_qiskit_bitstring

H_ising, ising_offset = qubo_to_ising(Q, offset)
print('SparsePauliOp (first 6 terms):')
for pauli, coeff in zip(H_ising.paulis[:6], H_ising.coeffs[:6]):
    print(f'  {paeff:.4f} * {pauli}' if False else f'  {float(coeff.real):.4f} * {str(pauli)}')
print(f'  ... ({len(H_ising.paulis)} terms total)')
print(f'Ising offset : {ising_offset:.4f}')

### Cross-check: Ising diagonal == QUBO energy for all bitstrings (9-qubit)

In [ ]:
from qiskit.quantum_info import Statevector
D = 9
mismatches = []
for idx in range(2**D):
    bit_str = format(idx, f'0{D}b')
    # QUBO little-endian decoding
    x_vec = np.array([int(bit_str[D-1-k]) for k in range(D)], dtype=int)
    qubo_e = evaluate_qubo_energy(Q, offset, x_vec)
    ising_e = float(H_ising.expectation_value(Statevector.from_label(bit_str)).real)
    if abs(qubo_e - ising_e) > 1e-6:
        mismatches.append((bit_str, qubo_e, ising_e))
print(f'Checked {2**D} bitstrings. Mismatches: {len(mismatches)}')
if len(mismatches) == 0:
    print('✓  Ising diagonal matches QUBO energy for all 512 states.')

## 4 · QAOA Circuit

```
|ψ₀⟩ = |+⟩^⊗N  (uniform superposition)
For layer ℓ = 1..p:
  exp(-iγ_ℓ H_C)   — cost unitary
  exp(-iβ_ℓ Σ X_k) — X-mixer
Measure
```

Optimized with COBYLA (3 random restarts).

In [ ]:
from qiskit.circuit.library import QAOAAnsatz
from qiskit import transpile

ansatz = QAOAAnsatz(cost_operator=H_ising, reps=2)
t_circ = transpile(ansatz, basis_gates=['h', 'rx', 'rz', 'cx'])
print(f'Qubits          : {t_circ.num_qubits}')
print(f'Parameters      : {len(t_circ.parameters)}')
print(f'Depth (p=2)     : {t_circ.depth()}')
print(f'CX gate count   : {t_circ.count_ops().get("cx", 0)}')

# Draw first few lines of circuit (compact)
print('\nCircuit (fold=80):')
print(t_circ.draw(output='text', fold=80))

## 5 · Run QAOA Optimization (p = 1 and p = 2)

In [ ]:
from solvers.qaoa import run_qaoa, plot_qaoa_probabilities
import time

opt_cost = h_res3['total_cost']
results = {}

for p_val in [1, 2]:
    print(f'Running QAOA p={p_val} (3 restarts, 80 iters)...', end=' ', flush=True)
    t0 = time.perf_counter()
    res = run_qaoa(Q, offset, N=3, p=p_val,
                   cvar_alpha=None, shots=4096, seed=42,
                   n_restarts=3, maxiter=80, optimal_cost=opt_cost)
    print(f'done ({time.perf_counter()-t0:.1f}s)')
    results[f'p={p_val}'] = res
    print(f'  Approx ratio     : {res["approx_ratio"]:.4f}')
    print(f'  Optimal-hit prob : {res["optimal_hit_prob"]*100:.2f}%')
    print(f'  Feasibility rate : {res["feasibility_rate"]*100:.2f}%')
    print(f'  Best feasible E  : {res["best_feasible_energy"]}')
    print()

In [ ]:
fig = plot_qaoa_probabilities(results['p=2'], top_k=14)
plt.tight_layout()
plt.show()

## 6 · CVaR-QAOA (α = 0.20)

Instead of optimizing mean energy, optimize the mean of the **best 20%** of samples.

*Reference: Barkoutsos et al., Quantum 4, 256 (2020)*

In [ ]:
print('Running CVaR-QAOA p=2 alpha=0.20...', end=' ', flush=True)
t0 = time.perf_counter()
res_cvar = run_qaoa(Q, offset, N=3, p=2,
                    cvar_alpha=0.20, shots=4096, seed=42,
                    n_restarts=3, maxiter=80, optimal_cost=opt_cost)
print(f'done ({time.perf_counter()-t0:.1f}s)')
print(f'  Approx ratio     : {res_cvar["approx_ratio"]:.4f}')
print(f'  Optimal-hit prob : {res_cvar["optimal_hit_prob"]*100:.2f}%')
print(f'  Feasibility rate : {res_cvar["feasibility_rate"]*100:.2f}%')

## 7 · Simulated Annealing (Classical QUBO Heuristic)

In [ ]:
from solvers.sa import solve_simulated_annealing

sa_res = solve_simulated_annealing(Q, offset, 3, seed=42)
print(f'SA best energy  : {sa_res["best_energy"]:.4f}')
print(f'SA feasible?    : {sa_res["is_feasible"]}')
print(f'SA cost (feas.) : {sa_res["best_feasible_energy"]}')
print(f'Runtime         : {sa_res["runtime_sec"]:.3f}s')

## 8 · Full Benchmark Comparison (3×3, 10 seeds)

Running a quick 10-instance subset for illustration. Full benchmark (30 seeds) is in `run_all.py`.

In [ ]:
from experiments.benchmark import run_benchmarks, plot_benchmark_figures

df = run_benchmarks(n_instances_3x3=10, n_instances_4x4=0,
                    run_qaoa_solvers=True, seed_offset=100)

summary = df.groupby('solver')['approx_ratio'].agg(['mean','std']).round(4)
print(summary.to_string())

In [ ]:
import pandas as pd

fig, ax = plt.subplots(figsize=(9, 4.5))
solver_order = [
    'Hungarian (Exact)', 'Simulated Annealing (QUBO)',
    'Severity-First Greedy', 'CVaR-QAOA (p=2, a=0.2)',
    'QAOA (p=2)', 'QAOA (p=1)', 'Nearest-Vehicle Greedy'
]
s = df.groupby('solver')['approx_ratio'].agg(['mean','std']).reindex(
    [s for s in solver_order if s in df['solver'].unique()]
).reset_index()

colors = ['#2ecc71','#3498db','#9b59b6','#e67e22','#f39c12','#f1c40f','#e74c3c']
ax.bar(s['solver'], s['mean'], yerr=s['std'].fillna(0),
       capsize=4, color=colors[:len(s)], edgecolor='black', linewidth=1.2)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=1.5, label='Optimal (1.0)')
ax.set_xticklabels(s['solver'], rotation=25, ha='right', fontsize=8.5)
ax.set_ylabel('Approx Ratio (lower = better)', fontweight='bold')
ax.set_title('Solver Comparison — 3×3 Instances (Simulator)', fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 9 · Honest Limitations

1. **Assignment is polynomial.** Hungarian is O(N³) — always wins here.
2. **Simulator only.** 9–16 qubits on Qiskit Statevector. No real hardware.
3. **X-mixer violates constraints** → some samples are infeasible.
4. **Qubit scaling:** N² qubits. City-scale dispatch needs decomposition.
5. **Synthetic grid** — no real traffic or hospital routing.

**Why it matters:** a correct, validated QUBO+QAOA pipeline for *harder* NP-hard dispatch variants (multi-route, capacity constraints) that future quantum devices could tackle.

---
*All results labelled **Simulator** — Qiskit Statevector / no real quantum hardware.*